# 🎯 Genie Code Maturity Assessment - Config Reader

## Purpose
This notebook reads the completed SDLC Assessment CSV/Excel and generates comprehensive maturity analysis including:
* Overall and phase-specific maturity scores
* Strengths and critical gaps identification  
* Visual dashboards (radar charts, heatmaps)
* Prioritized action plans
* Executive summary reports

## Input Files
**Option 1:** CSV - `SDLC_Assessment_Template.csv` (completed)
**Option 2:** Excel - `SDLC_Assessment_Interactive.xlsx` (completed)

## Workflow
1. **Load** - Read completed assessment
2. **Validate** - Check data quality
3. **Calculate** - Compute maturity scores
4. **Analyze** - Identify gaps and strengths
5. **Visualize** - Create charts and dashboards
6. **Report** - Generate executive summary
7. **Export** - Save results and action plans

---

*Run all cells in sequence to generate complete assessment report*

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("="*100)
print("STEP 1: LOAD ASSESSMENT CONFIGURATION")
print("="*100)

# Configuration - Update these paths if needed
CSV_PATH = '/Workspace/Users/sushant.mishriko@tigeranalytics.com/Genie Assessment Framework/SDLC_Assessment_Template.csv'
EXCEL_PATH = '/Workspace/Users/sushant.mishriko@tigeranalytics.com/Genie Assessment Framework/SDLC_Assessment_Interactive.xlsx'

# Try loading Excel first (preferred for client input), fallback to CSV
try:
    df_assessment = pd.read_excel(EXCEL_PATH, sheet_name='Assessment')
    source_file = 'Excel'
    print(f"\n✅ Loaded from Excel: {EXCEL_PATH}")
except:
    df_assessment = pd.read_csv(CSV_PATH)
    source_file = 'CSV'
    print(f"\n✅ Loaded from CSV: {CSV_PATH}")

print(f"\n📊 Assessment Data Loaded:")
print(f"  • Source: {source_file}")
print(f"  • Total Questions: {len(df_assessment)}")
print(f"  • Assessment Date: {datetime.now().strftime('%Y-%m-%d')}")

# Handle different column name variations
if 'Current_Score' not in df_assessment.columns and 'Score' in df_assessment.columns:
    df_assessment['Current_Score'] = df_assessment['Score']

print(f"\n📋 Phases Covered:")
for phase in df_assessment['Phase'].unique():
    count = len(df_assessment[df_assessment['Phase'] == phase])
    print(f"  • {phase}: {count} questions")

print("\n" + "="*100)

# Display first few rows
display(df_assessment[['Phase', 'Question_ID', 'Question', 'Current_Score']].head(5))

In [0]:
print("="*100)
print("STEP 2: VALIDATE ASSESSMENT DATA")
print("="*100)

# Convert Current_Score to numeric
df_assessment['Current_Score'] = pd.to_numeric(df_assessment['Current_Score'], errors='coerce')

# Check for missing scores
missing_scores = df_assessment[df_assessment['Current_Score'].isna()]
scored_count = len(df_assessment) - len(missing_scores)

print(f"\n📊 Validation Results:")
print(f"  • Total questions: {len(df_assessment)}")
print(f"  • Questions with scores: {scored_count} ({scored_count/len(df_assessment)*100:.1f}%)")
print(f"  • Questions missing scores: {len(missing_scores)} ({len(missing_scores)/len(df_assessment)*100:.1f}%)")

if len(missing_scores) > 0:
    print(f"\n⚠️ WARNING: {len(missing_scores)} questions are missing scores:")
    for idx, row in missing_scores.head(10).iterrows():
        print(f"  • {row['Question_ID']}: {row['Question'][:70]}...")
    if len(missing_scores) > 10:
        print(f"  ... and {len(missing_scores) - 10} more")
    print("\n‼️ Please complete all scores for accurate assessment.")
else:
    print("\n✅ All questions have been scored!")

# Validate score range (1-5)
invalid_scores = df_assessment[
    (df_assessment['Current_Score'].notna()) &
    ((df_assessment['Current_Score'] < 1) | (df_assessment['Current_Score'] > 5))
]

if len(invalid_scores) > 0:
    print(f"\n⚠️ WARNING: {len(invalid_scores)} questions have invalid scores (must be 1-5):")
    for idx, row in invalid_scores.iterrows():
        print(f"  • {row['Question_ID']}: Score = {row['Current_Score']}")
else:
    print("\n✅ All scores are within valid range (1-5)!")

# Score distribution
print(f"\n📊 Score Distribution:")
for score in range(1, 6):
    count = len(df_assessment[df_assessment['Current_Score'] == score])
    percentage = count / scored_count * 100 if scored_count > 0 else 0
    bar = '█' * int(percentage / 2)
    print(f"  Level {score}: {count:2d} questions ({percentage:5.1f}%) [{bar}]")

print("\n" + "="*100)

# Filter out invalid/missing scores for analysis
df_valid = df_assessment[df_assessment['Current_Score'].between(1, 5, inclusive='both')].copy()
print(f"\n✅ Proceeding with {len(df_valid)} valid scored questions for analysis")

In [0]:
print("="*100)
print("STEP 3: CALCULATE MATURITY SCORES")
print("="*100)

# Define maturity level function
def get_maturity_level(score):
    if pd.isna(score):
        return "Not Assessed"
    elif score < 1.5:
        return "Level 1: Initial/Ad-hoc"
    elif score < 2.5:
        return "Level 2: Aware/Experimental"
    elif score < 3.5:
        return "Level 3: Defined/Structured"
    elif score < 4.5:
        return "Level 4: Managed/Optimized"
    else:
        return "Level 5: Innovative/Leading"

# Calculate overall score
overall_score = df_valid['Current_Score'].mean()
overall_maturity = get_maturity_level(overall_score)

print(f"\n🎯 OVERALL MATURITY")
print(f"  • Average Score: {overall_score:.2f} / 5.00")
print(f"  • Maturity Level: {overall_maturity}")
print(f"  • Based on {len(df_valid)} assessed questions")

# Calculate phase scores
phase_scores = df_valid.groupby('Phase')['Current_Score'].agg([
    ('Average_Score', 'mean'),
    ('Min_Score', 'min'),
    ('Max_Score', 'max'),
    ('Question_Count', 'count')
]).round(2)

phase_scores['Maturity_Level'] = phase_scores['Average_Score'].apply(get_maturity_level)

print(f"\n📋 PHASE-SPECIFIC SCORES")
print("-" * 100)
for phase, row in phase_scores.sort_values('Average_Score').iterrows():
    bar_length = int(row['Average_Score'] * 10)
    bar = '█' * bar_length + '░' * (50 - bar_length)
    print(f"\n{phase}")
    print(f"  [{bar}] {row['Average_Score']:.2f}/5.00")
    print(f"  • Maturity: {row['Maturity_Level']}")
    print(f"  • Range: {row['Min_Score']:.1f} - {row['Max_Score']:.1f} | Questions: {int(row['Question_Count'])}")

# Identify priority phases (lowest scores)
top_priority_phases = phase_scores.sort_values('Average_Score').head(3)
print(f"\n🔍 TOP 3 PRIORITY PHASES (Lowest Scores)")
print("-" * 100)
for idx, (phase, row) in enumerate(top_priority_phases.iterrows(), 1):
    print(f"{idx}. {phase}: {row['Average_Score']:.2f}/5.00")

print("\n" + "="*100)

# Store results for later use
assessment_results = {
    'overall_score': overall_score,
    'overall_maturity': overall_maturity,
    'phase_scores': phase_scores,
    'assessment_date': datetime.now().strftime('%Y-%m-%d'),
    'total_questions': len(df_valid)
}

# Display phase summary
display(phase_scores.sort_values('Average_Score'))

In [0]:
print("="*100)
print("STEP 4: GAP ANALYSIS - IDENTIFY STRENGTHS AND CRITICAL GAPS")
print("="*100)

# Identify strengths (score >= 3.5)
strengths = df_valid[df_valid['Current_Score'] >= 3.5].sort_values('Current_Score', ascending=False)

# Identify critical gaps (score <= 2)
critical_gaps = df_valid[df_valid['Current_Score'] <= 2].sort_values('Current_Score')

# Identify medium priority gaps (score 2-3.5)
medium_gaps = df_valid[
    (df_valid['Current_Score'] > 2) & 
    (df_valid['Current_Score'] < 3.5)
].sort_values('Current_Score')

print(f"\n✅ STRENGTHS (Score >= 3.5): {len(strengths)} questions")
print("-" * 100)
if len(strengths) > 0:
    print("\nTop Strengths:")
    for idx, row in strengths.head(10).iterrows():
        print(f"  • {row['Question_ID']} (Score: {row['Current_Score']:.1f}): {row['Question'][:80]}...")
        print(f"    Phase: {row['Phase']}")
else:
    print("  No areas of strength yet - significant improvement opportunity across all phases!")

print(f"\n🔴 CRITICAL GAPS (Score <= 2): {len(critical_gaps)} questions")
print("-" * 100)
if len(critical_gaps) > 0:
    print("\nMost Critical Gaps (Immediate Action Required):")
    for idx, row in critical_gaps.head(15).iterrows():
        print(f"  • {row['Question_ID']} (Score: {row['Current_Score']:.1f}): {row['Question'][:80]}...")
        print(f"    Phase: {row['Phase']}")
        if 'Gap_Description' in df_valid.columns and pd.notna(row['Gap_Description']):
            gap_desc = str(row['Gap_Description'])[:100]
            if gap_desc.strip():
                print(f"    Gap: {gap_desc}...")
        if 'Priority' in df_valid.columns and pd.notna(row['Priority']):
            print(f"    Priority: {row['Priority']}")
else:
    print("  No critical gaps - excellent foundation!")

print(f"\n🟡 MEDIUM PRIORITY GAPS (Score 2-3.5): {len(medium_gaps)} questions")
print("-" * 100)
if len(medium_gaps) > 0:
    print(f"\nTop {min(10, len(medium_gaps))} Medium Priority Improvements:")
    for idx, row in medium_gaps.head(10).iterrows():
        print(f"  • {row['Question_ID']} (Score: {row['Current_Score']:.1f}): {row['Question'][:80]}...")
        print(f"    Phase: {row['Phase']}")

print("\n" + "="*100)

# Summary statistics
print(f"\n📊 Gap Analysis Summary:")
print(f"  • Strengths (>=3.5): {len(strengths)} ({len(strengths)/len(df_valid)*100:.1f}%)")
print(f"  • Critical Gaps (<=2): {len(critical_gaps)} ({len(critical_gaps)/len(df_valid)*100:.1f}%)")
print(f"  • Medium Gaps (2-3.5): {len(medium_gaps)} ({len(medium_gaps)/len(df_valid)*100:.1f}%)")

# Create gap summary DataFrames for export
gaps_summary = pd.concat([
    critical_gaps[['Phase', 'Question_ID', 'Question', 'Current_Score', 'Priority']].assign(Gap_Type='Critical'),
    medium_gaps[['Phase', 'Question_ID', 'Question', 'Current_Score', 'Priority']].assign(Gap_Type='Medium')
]).sort_values('Current_Score')

strengths_summary = strengths[['Phase', 'Question_ID', 'Question', 'Current_Score', 'Genie_Capability']]

print(f"\n✅ Gap analysis complete - {len(gaps_summary)} improvement opportunities identified")

In [0]:
print("="*100)
print("STEP 5: VISUALIZATION - RADAR CHART BY PHASE")
print("="*100)

print("\n📊 Creating radar chart for phase maturity scores...\n")

# Prepare data for radar chart
phases = phase_scores.index.tolist()
scores = phase_scores['Average_Score'].tolist()

# Number of variables
num_vars = len(phases)

# Compute angle for each axis
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
scores_plot = scores + scores[:1]  # Complete the circle
angles_plot = angles + angles[:1]

# Create figure with larger size
fig, ax = plt.subplots(figsize=(14, 14), subplot_kw=dict(projection='polar'))

# Draw the current score
ax.plot(angles_plot, scores_plot, 'o-', linewidth=3, label='Current Score', color='#1f77b4', markersize=10)
ax.fill(angles_plot, scores_plot, alpha=0.25, color='#1f77b4')

# Add target level lines
target_level3 = [3.0] * (num_vars + 1)
ax.plot(angles_plot, target_level3, '--', linewidth=2, label='Target Level 3', color='green', alpha=0.7)

target_level4 = [4.0] * (num_vars + 1)
ax.plot(angles_plot, target_level4, '--', linewidth=2, label='Target Level 4', color='orange', alpha=0.7)

# Fix axis to go in the right order and start at 12 o'clock
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# Set labels and formatting
ax.set_xticks(angles)
ax.set_xticklabels(phases, size=10, weight='bold')
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1\n(Initial)', '2\n(Aware)', '3\n(Defined)', '4\n(Managed)', '5\n(Leading)'], size=9)
ax.set_rlabel_position(0)

# Add gridlines
ax.grid(True, linestyle='--', alpha=0.7, linewidth=1.5)

# Add title and legend
title_text = f'Genie Code Maturity Assessment\nOverall Score: {overall_score:.2f}/5.00 - {overall_maturity}\nAssessment Date: {assessment_results["assessment_date"]}'
plt.title(title_text, size=16, fontweight='bold', pad=30)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=11)

plt.tight_layout()
plt.show()

print("✅ Radar chart created successfully!")
print("\n📊 Chart Insights:")
print(f"  • Phases above Level 3 (green): {len([s for s in scores if s >= 3.0])}")
print(f"  • Phases below Level 2: {len([s for s in scores if s < 2.0])}")
print(f"  • Gap to Level 4: {4.0 - overall_score:.2f} points")
print("="*100)

In [0]:
print("="*100)
print("STEP 6: VISUALIZATION - HEATMAP OF QUESTIONS BY PHASE")
print("="*100)

print("\n📊 Creating heatmap of question scores by phase...\n")

# Create pivot table for heatmap
heatmap_data = df_valid.pivot_table(
    values='Current_Score',
    index='Question_ID',
    columns='Phase',
    aggfunc='mean'
)

# Sort by phase average
phase_order = phase_scores.sort_values('Average_Score').index.tolist()
heatmap_data = heatmap_data[phase_order]

# Create figure
fig, ax = plt.subplots(figsize=(16, max(12, len(df_valid) * 0.3)))

# Create heatmap
sns.heatmap(
    heatmap_data.T,
    annot=False,
    cmap='RdYlGn',
    center=3.0,
    vmin=1,
    vmax=5,
    cbar_kws={'label': 'Maturity Score'},
    linewidths=0.5,
    ax=ax
)

ax.set_title(f'Genie Code Maturity Heatmap - Question Scores by Phase\nOverall Score: {overall_score:.2f}/5.00', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Question ID', fontsize=12, fontweight='bold')
ax.set_ylabel('Phase', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Heatmap created successfully!")
print("\n🎨 Color Legend:")
print("  • Red: Low scores (1-2.5) - Critical gaps")
print("  • Yellow: Medium scores (2.5-3.5) - Improvement needed")
print("  • Green: High scores (3.5-5) - Strengths")
print("="*100)

In [0]:
print("="*100)
print("STEP 7: GENERATE PRIORITIZED ACTION PLAN")
print("="*100)

print("\n🛠️ Building prioritized action plan based on gaps...\n")

# Build action plan from critical and medium gaps
action_items = []

# Add critical gaps (Score <= 2) with HIGH priority
for idx, row in critical_gaps.iterrows():
    priority = row.get('Priority', 'High')
    if pd.isna(priority) or str(priority).strip() == '':
        priority = 'High'
    
    action_items.append({
        'Priority': priority,
        'Phase': row['Phase'],
        'Question_ID': row['Question_ID'],
        'Current_Score': row['Current_Score'],
        'Gap_Type': 'Critical',
        'Question': row['Question'],
        'Genie_Capability': row['Genie_Capability'],
        'Gap_Description': row.get('Gap_Description', 'N/A'),
        'Recommended_Action': f"Implement {row['Genie_Capability']} to move from Level {int(row['Current_Score'])} to Level 3"
    })

# Add medium priority gaps (Score 2-3.5)
for idx, row in medium_gaps.head(15).iterrows():  # Limit to top 15 medium gaps
    priority = row.get('Priority', 'Medium')
    if pd.isna(priority) or str(priority).strip() == '':
        priority = 'Medium'
    
    action_items.append({
        'Priority': priority,
        'Phase': row['Phase'],
        'Question_ID': row['Question_ID'],
        'Current_Score': row['Current_Score'],
        'Gap_Type': 'Medium',
        'Question': row['Question'],
        'Genie_Capability': row['Genie_Capability'],
        'Gap_Description': row.get('Gap_Description', 'N/A'),
        'Recommended_Action': f"Enhance {row['Genie_Capability']} to reach Level 4"
    })

# Create action plan DataFrame
action_plan_df = pd.DataFrame(action_items)

# Sort by priority and score
priority_order = {'High': 1, 'Medium': 2, 'Low': 3}
action_plan_df['Priority_Rank'] = action_plan_df['Priority'].map(priority_order)
action_plan_df = action_plan_df.sort_values(['Priority_Rank', 'Current_Score', 'Phase'])
action_plan_df = action_plan_df.drop('Priority_Rank', axis=1)

print(f"✅ Action plan generated with {len(action_plan_df)} items")
print(f"\n📊 Action Plan Breakdown:")
print(f"  • High Priority: {len(action_plan_df[action_plan_df['Priority'] == 'High'])} items")
print(f"  • Medium Priority: {len(action_plan_df[action_plan_df['Priority'] == 'Medium'])} items")
print(f"  • Low Priority: {len(action_plan_df[action_plan_df['Priority'] == 'Low'])} items")

print("\n" + "-" * 100)
print("TOP 10 ACTION ITEMS")
print("-" * 100)

for idx, row in action_plan_df.head(10).iterrows():
    print(f"\n{idx + 1}. [{row['Priority']} Priority] {row['Question_ID']} - Score: {row['Current_Score']:.1f}")
    print(f"   Phase: {row['Phase']}")
    print(f"   Question: {row['Question'][:100]}...")
    print(f"   Action: {row['Recommended_Action']}")

print("\n" + "="*100)

# Display action plan table
print("\n📋 Full Action Plan Table:")
display(action_plan_df[['Priority', 'Phase', 'Question_ID', 'Current_Score', 'Gap_Type', 'Recommended_Action']].head(20))

In [0]:
print("="*100)
print("EXECUTIVE SUMMARY REPORT")
print("="*100)

print(f"\n📄 GENIE CODE MATURITY ASSESSMENT")
print(f"Assessment Date: {assessment_results['assessment_date']}")
print(f"Total Questions Assessed: {assessment_results['total_questions']}")

print(f"\n\n🎯 OVERALL MATURITY")
print("-" * 100)
print(f"Average Maturity Score: {overall_score:.2f} / 5.00")
print(f"Maturity Level: {overall_maturity}")

# Visual bar representation
bar_filled = int(overall_score * 10)
bar_empty = 50 - bar_filled
print(f"\nProgress: [█{'=' * bar_filled}{' ' * bar_empty}] {overall_score/5*100:.1f}%")

print(f"\n\n📊 MATURITY LEVEL DISTRIBUTION")
print("-" * 100)
for level in range(5, 0, -1):
    count = len(df_valid[df_valid['Current_Score'].between(level-0.5, level+0.5, inclusive='left')])
    pct = count / len(df_valid) * 100
    bar = '█' * int(pct / 2)
    print(f"Level {level}: {count:2d} questions ({pct:5.1f}%) [{bar}]")

print(f"\n\n📋 TOP 3 STRONGEST PHASES")
print("-" * 100)
for idx, (phase, row) in enumerate(phase_scores.sort_values('Average_Score', ascending=False).head(3).iterrows(), 1):
    print(f"{idx}. {phase}")
    print(f"   Score: {row['Average_Score']:.2f}/5.00 ({row['Maturity_Level']})")
    print(f"   Questions: {int(row['Question_Count'])} | Range: {row['Min_Score']:.1f}-{row['Max_Score']:.1f}")

print(f"\n\n🔍 TOP 3 PRIORITY PHASES (Need Improvement)")
print("-" * 100)
for idx, (phase, row) in enumerate(phase_scores.sort_values('Average_Score').head(3).iterrows(), 1):
    print(f"{idx}. {phase}")
    print(f"   Score: {row['Average_Score']:.2f}/5.00 ({row['Maturity_Level']})")
    print(f"   Questions: {int(row['Question_Count'])} | Range: {row['Min_Score']:.1f}-{row['Max_Score']:.1f}")
    gap_to_level3 = max(0, 3.0 - row['Average_Score'])
    print(f"   Gap to Level 3: {gap_to_level3:.2f} points")

print(f"\n\n✅ STRENGTHS (Score >= 3.5)")
print("-" * 100)
print(f"Total Strengths: {len(strengths)} questions ({len(strengths)/len(df_valid)*100:.1f}%)")
if len(strengths) > 0:
    print("\nTop 5 Strengths:")
    for idx, row in strengths.head(5).iterrows():
        print(f"  • {row['Question_ID']}: {row['Question'][:70]}... (Score: {row['Current_Score']:.1f})")

print(f"\n\n🔴 CRITICAL GAPS (Score <= 2)")
print("-" * 100)
print(f"Total Critical Gaps: {len(critical_gaps)} questions ({len(critical_gaps)/len(df_valid)*100:.1f}%)")
if len(critical_gaps) > 0:
    print("\nTop 5 Critical Gaps (Immediate Action Required):")
    for idx, row in critical_gaps.head(5).iterrows():
        print(f"  • {row['Question_ID']}: {row['Question'][:70]}... (Score: {row['Current_Score']:.1f})")

print(f"\n\n🎯 RECOMMENDED NEXT STEPS")
print("-" * 100)
print(f"1. Address {len(critical_gaps)} critical gaps to establish baseline capabilities")
print(f"2. Focus on top 3 priority phases: {', '.join(phase_scores.sort_values('Average_Score').head(3).index.tolist())}")
print(f"3. Leverage {len(strengths)} existing strengths as foundation for expansion")
print(f"4. Target Level 3 (Defined) as near-term goal - requires {max(0, 3.0 - overall_score):.2f} point improvement")
print(f"5. Implement action plan with {len(action_plan_df[action_plan_df['Priority'] == 'High'])} high-priority items")

print("\n" + "="*100)
print("\n✅ Executive summary complete!")

In [0]:
print("="*100)
print("STEP 9: EXPORT RESULTS AND REPORTS")
print("="*100)

print("\n💾 Exporting assessment results to CSV files...\n")

# Define export directory
export_dir = '/Workspace/Users/sushant.mishrako@tigeranalytics.com/Genie Assessment Framework/'
assessment_date_str = assessment_results['assessment_date'].replace('-', '')

# 1. Export Phase Scores Summary
phase_scores_export = phase_scores.reset_index()
phase_scores_file = f"{export_dir}Phase_Scores_{assessment_date_str}.csv"
phase_scores_export.to_csv(phase_scores_file, index=False)
print(f"✅ Phase scores exported: {phase_scores_file}")

# 2. Export Action Plan
action_plan_file = f"{export_dir}Action_Plan_{assessment_date_str}.csv"
action_plan_df.to_csv(action_plan_file, index=False)
print(f"✅ Action plan exported: {action_plan_file}")

# 3. Export Gaps Summary
if len(gaps_summary) > 0:
    gaps_file = f"{export_dir}Gaps_Summary_{assessment_date_str}.csv"
    gaps_summary.to_csv(gaps_file, index=False)
    print(f"✅ Gaps summary exported: {gaps_file}")

# 4. Export Strengths Summary
if len(strengths_summary) > 0:
    strengths_file = f"{export_dir}Strengths_Summary_{assessment_date_str}.csv"
    strengths_summary.to_csv(strengths_file, index=False)
    print(f"✅ Strengths summary exported: {strengths_file}")

# 5. Export Executive Summary as text file
exec_summary_file = f"{export_dir}Executive_Summary_{assessment_date_str}.txt"
with open(exec_summary_file, 'w') as f:
    f.write("=" * 100 + "\n")
    f.write("GENIE CODE MATURITY ASSESSMENT - EXECUTIVE SUMMARY\n")
    f.write("=" * 100 + "\n\n")
    f.write(f"Assessment Date: {assessment_results['assessment_date']}\n")
    f.write(f"Total Questions: {assessment_results['total_questions']}\n\n")
    f.write(f"OVERALL MATURITY\n")
    f.write("-" * 100 + "\n")
    f.write(f"Score: {overall_score:.2f} / 5.00\n")
    f.write(f"Level: {overall_maturity}\n\n")
    f.write(f"PHASE SCORES\n")
    f.write("-" * 100 + "\n")
    for phase, row in phase_scores.sort_values('Average_Score').iterrows():
        f.write(f"{phase}: {row['Average_Score']:.2f}/5.00 ({row['Maturity_Level']})\n")
    f.write(f"\nSTRENGTHS: {len(strengths)} questions\n")
    f.write(f"CRITICAL GAPS: {len(critical_gaps)} questions\n")
    f.write(f"MEDIUM GAPS: {len(medium_gaps)} questions\n")

print(f"✅ Executive summary exported: {exec_summary_file}")

print("\n" + "="*100)
print("\n✅ ALL EXPORTS COMPLETE!")
print("\n📊 Generated Files:")
print(f"  1. Phase Scores: Phase_Scores_{assessment_date_str}.csv")
print(f"  2. Action Plan: Action_Plan_{assessment_date_str}.csv")
print(f"  3. Gaps Summary: Gaps_Summary_{assessment_date_str}.csv")
print(f"  4. Strengths Summary: Strengths_Summary_{assessment_date_str}.csv")
print(f"  5. Executive Summary: Executive_Summary_{assessment_date_str}.txt")
print("\n📋 Files saved to: {}".format(export_dir))
print("="*100)

# Display final summary tables
print("\n📊 FINAL SUMMARY TABLES")
print("\n1. Phase Scores:")
display(phase_scores_export)

print("\n2. Top 10 Action Items:")
display(action_plan_df[['Priority', 'Phase', 'Question_ID', 'Current_Score', 'Recommended_Action']].head(10))

# ✅ Assessment Analysis Complete!

## 📊 What Was Generated

### Reports & Visualizations
1. **Overall Maturity Score** - Single metric showing organizational maturity level
2. **Phase-Specific Scores** - Breakdown across 9 SDLC phases
3. **Radar Chart** - Visual representation of maturity by phase
4. **Heatmap** - Detailed question-level scores
5. **Gap Analysis** - Strengths and critical improvement areas
6. **Prioritized Action Plan** - Concrete next steps organized by priority
7. **Executive Summary** - High-level report for stakeholders

### Exported Files
All results exported to: `/Workspace/Users/sushant.mishrako@tigeranalytics.com/Genie Assessment Framework/`
- `Phase_Scores_YYYYMMDD.csv`
- `Action_Plan_YYYYMMDD.csv`
- `Gaps_Summary_YYYYMMDD.csv`
- `Strengths_Summary_YYYYMMDD.csv`
- `Executive_Summary_YYYYMMDD.txt`

---

## 🎯 Recommended Next Steps

### 1. Review & Prioritize
* Share executive summary with stakeholders
* Review action plan with implementation teams
* Validate priority levels with business owners

### 2. Plan Implementation Waves
* **Wave 1 (Quick Wins)**: Address high-priority gaps with score ≤ 2
* **Wave 2 (Foundation)**: Build systematic processes for scores 2-3
* **Wave 3 (Optimization)**: Enhance capabilities for scores 3-4
* **Wave 4 (Innovation)**: Implement advanced features for scores 4-5

### 3. Re-Assessment Schedule
* **Monthly**: Track progress on action items
* **Quarterly**: Re-run full assessment to measure improvement
* **Annually**: Comprehensive review and strategy adjustment

---

## 📝 How to Use This Notebook

### For Initial Assessment
1. Complete the `SDLC_Assessment_Interactive.xlsx` file
2. Save completed file to framework folder
3. Run this notebook (Run All)
4. Review outputs and share reports

### For Progress Tracking
1. Update scores in assessment file based on improvements
2. Re-run this notebook
3. Compare results with previous assessment
4. Adjust action plan based on progress

### For Custom Analysis
* Modify visualizations (cells 5-6) for different chart types
* Adjust gap thresholds (cell 4) to change priority definitions
* Filter action plan (cell 7) by specific phases or priorities
* Export additional custom reports (cell 9)

---

## 🛠️ Customization Options

**Maturity Thresholds** (Cell 3):
```python
# Modify these ranges to adjust level definitions
Level 1: < 1.5
Level 2: 1.5 - 2.5
Level 3: 2.5 - 3.5
Level 4: 3.5 - 4.5
Level 5: ≥ 4.5
```

**Gap Priorities** (Cell 4):
```python
Critical: score ≤ 2.0
Medium: 2.0 < score < 3.5
Low: score ≥ 3.5
```

**Export Formats**:
* Add JSON export for API integration
* Generate PDF reports using reportlab
* Create PowerPoint slides using python-pptx

---

## 📞 Support & Questions

For assistance with:
* Assessment completion
* Results interpretation
* Custom analysis requests
* Implementation guidance

Contact the Genie Code Assessment Team

---

**Last Updated:** 2026-04-07  
**Framework Version:** 2.0  
**Questions:** 47 across 9 SDLC phases